# R-tag pipeline — single run walkthrough

This notebook runs the **Python R-tag (RTEG) workflow** end-to-end for the KB331 sample filter, showcasing what each script and input file does.

**Ground truth from SKILL script by Jing Yang:** manual flow in Virtuoso via `rdsBawTEGAutoFromTemp.il`  
**Docs:** [`workflow.md`](workflow.md) · [`README.md`](README.md) · [`../CLAUDE.md`](../CLAUDE.md)

| Phase | Scripts | Purpose |
|---|---|---|
| **Inputs** | GDS + layermap + JSON | Filter, frame, probe pad |
| **Foundation** | `layermap.py`, `inspect_refs.py`, `separate.py`, `rteg_skill.py`, `build_rteg.py` | Read GDS, identify resonators, visual export |
| **Prepare** | `prepare_rteg.py`, `inspect_golden.py`, `geometry.py` | Template assembly — frame + centered ppd + centered resonator (all 8) |
| **Route (v1)** | `route_rteg.py` | Signal route + ground recut + DRC (S3 demo) |

Run all cells top-to-bottom from the `python_code/` directory (or adjust `ROOT` below).

Workflow Steps:
1. Process Inputs (layermap, filter GDS file, die frame, GSG ppd frame)
2. Selection: take out resonators from GDS file 
3. Separation: for each resonator center it in GSG ppd frame
4. Setting up: Place ppd + resonator frame in the center of die frame
5. Routing: Start MBE and MTE Routes
6. DRC Verification

In [1]:
from __future__ import annotations
import io
import json
import sys
import warnings
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

Working directory: C:\Users\santosr4\Documents\rTEG Automation\python_code
Input files:       C:\Users\santosr4\Documents\rTEG Automation\python_code\input_files
Draft output:      C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output
Source code:       C:\Users\santosr4\Documents\rTEG Automation\python_code\src


## Define Inputs Here For Running This Notebook

In [2]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process Inputs

In [3]:
# Ensure all referenced input files exist, report and handle missing files

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,  # will fill after loading FRAME sizing
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

file_errors = []

# Check all files, prepare info for table rows
rows = []
frame_size_str = "unknown size"

# Check supporting files first so sizing is available for FRAME
for label, path in input_files.items():
    exists = path.exists()
    if not exists:
        file_errors.append(f"ERROR: {label} file not found: {path}")
    # Calculate size
    size = f"{path.stat().st_size:,} B" if exists and path.is_file() else "—"
    # Frame sizing to be updated after loading below
    rows.append({"file": label, "path": path.name, "exists": exists, "size": size, "role": input_roles[label]})

# Specifically for FRAME, if it is missing, output an error and skip sizing attempts
if FRAME.exists():
    try:
        frame_lib = gdstk.read_gds(FRAME)
        frame_cell = frame_lib.top_level()[0]
        frame_bb = frame_cell.bounding_box()
        if frame_bb is not None:
            (fx0, fy0), (fx1, fy1) = frame_bb
            frame_width = fx1 - fx0
            frame_height = fy1 - fy0
            frame_size_str = f"{frame_width:.1f}×{frame_height:.1f} µm"
    except Exception as e:
        file_errors.append(f"ERROR: Unable to read Frame template or extract bounding box: {e}")

# Update the row for FRAME with computed size string
for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

# If there were errors, print them, otherwise continue
if file_errors:
    for err in file_errors:
        print(err)
    print("❌ Aborting due to missing required input files ❌")
else:
    display(pd.DataFrame(rows))

,file,path,exists,size,role
0,Filter layout,KB331_N_01.gds,True,"655,360 B",Clean hierarchical filter GDS — resonators + c...
1,Frame template,KB331_N_Frame.gds,True,"34,816 B",460.0×580.0 µm GSG probe frame (six BAW_MB2 pads)
2,Probe device,GSG_frame.gds,True,"10,240 B",ppd_1port — GSG pad reference
3,Layer table,layermap,True,"3,971 B","Skyworks name → GDS (layer, datatype)"


## 2. Selection

2.1 start with layermap definitions

2.2 inspect layer references

2.3 start identifying resonators from filter GDS file

### 2.1 `layermap.py` — layer name lookups

Loads the Skyworks layermap file from `LAYERMAP` (defined above). Maps names like `BAW_MBE` to GDS `(layer, datatype)` pairs.

In [ ]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

Loaded 155 layers from layermap
  BAW_MBE      -> GDS (2, 0)
  BAW_MTE      -> GDS (5, 0)
  BAW_MB2      -> GDS (12, 0)
  BAW_EDGE     -> GDS (9, 0)


### 2.2 `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [ ]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")


=== KB331_N_01.gds ===

shuntq3_CDNS_781040784740: 80 polys, bbox (-127.4, -49.5)-(128.1, 54.5) (no references)
   labels (9):
     'P1' @ (0.0, -0.0)  layer=(5,0)
     'P2' @ (0.0, -0.0)  layer=(2,0)
     'freq=1478M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 9.5)  layer=(221,0)
     '[@instanceName]' @ (0.0, 36.0)  layer=(221,0)
     'ReW=-99' @ (0.0, 1.5)  layer=(221,0)
     'MRaW=2.2' @ (0.0, -6.5)  layer=(221,0)
     'MTE_CLEN=172.6' @ (0.0, -14.5)  layer=(221,0)
     'MBE_CLEN=377.9' @ (0.0, -22.5)  layer=(221,0)

seriesq3_CDNS_781040784741: 93 polys, bbox (-37.6, -53.2)-(43.3, 53.2) (no references)
   labels (9):
     'P1' @ (0.0, 0.0)  layer=(5,0)
     'P2' @ (0.0, 0.0)  layer=(2,0)
     'freq=1541M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 5.0)  layer=(221,0)
     '[@instanceName]' @ (0.0, 10.0)  layer=(221,0)
     'ReW=1.8' @ (0.0, 0.0)  laye

### 2.3 `separate.py` — resonator and vtb identification

Finds placed resonators (masters: `series`, `shunt`, `rcap`, `mimcap`) and `vtb` vias under the filter parent cell. Returns dataframe-ready rows via `identify(FILTER)`.

In [8]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows())
via_df = pd.DataFrame(identification.via_rows())

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

Parent cell: KB331_N_01
Resonators: 8  |  Vias: 4


,index,inst_name,master_name,type,origin_x,origin_y,rotation_deg,split_base
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,282.6,183.1,0.0,None
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,234.0,98.3,180.0,None
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,95.8,145.1,90.0,None
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,92.0,217.4,270.0,None
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,311.9,458.9,90.0,None
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,157.8,412.7,0.0,None
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,307.6,317.6,270.0,None
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,187.7,294.1,270.0,None


# START OF AUTOMATION Workflow 
everything above this is just independent scripts checking

## 5. `rteg_skill.py` — shared preprocessing helpers

Foundation is built in three steps before each resonator is placed:

1. **Die frame** (`KB331_N_Frame`) at top-left `(0, 0)`
2. **`ppd_1port`** centered inside the die frame bbox
3. **Resonator** shifted so its bbox center lands on the combined assembly center

Also provides inst-name inference and `connect_backup` loading.

In [ ]:
import tempfile

from IPython.display import SVG, display
from rteg_skill import (
    add_foundation_refs,
    bbox_center,
    build_foundation,
    frame_top_cell,
    infer_inst_names,
    load_connect_backup,
    placement_shift,
    resonator_world_bbox,
    rteg_cell_name,
)

# 1. Load die frame
frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_top_cell(frame_lib)

# 2. Load ppd and build foundation (ppd centered in frame)
ppd_lib = gdstk.read_gds(PPD)
ppd_cell = next(c for c in ppd_lib.cells if c.name == "ppd_1port")
foundation = build_foundation(frame_cell, ppd_cell)

# 3. Place resonator at assembly center (sample: index 6 / S3)
sample_res = res_list[6]
dx, dy = placement_shift(sample_res, frame_cell, ppd_cell, foundation=foundation)
res_center = bbox_center(resonator_world_bbox(sample_res))
shifted_res_center = (res_center[0] + dx, res_center[1] + dy)

inst_names = infer_inst_names(res_list, INST_MAP)
backup = load_connect_backup(parent, ROOT, filter_lib)

print(f"Frame origin: {foundation.frame_origin}")
print(f"PPD origin (centered in frame): {tuple(round(x, 1) for x in foundation.ppd_origin)}")
print(f"Assembly center: {tuple(round(x, 1) for x in foundation.assembly_center)}")
print(f"Resonator shift (dx, dy) for index 6: ({dx:.1f}, {dy:.1f})")
print(f"Resonator bbox center after shift: ({shifted_res_center[0]:.1f}, {shifted_res_center[1]:.1f})")
print(f"connect_backup loaded: {backup is not None}")
print(f"Overrides file: {json.loads(INST_MAP.read_text())}")

display(pd.DataFrame([
    {"index": i, "inferred_inst": inst_names[i], "cell_name": rteg_cell_name(parent, inst_names[i])}
    for i in range(len(res_list))
]))


def preview_rteg_cell(res, inst_name: str) -> gdstk.Cell:
    """In-memory top cell: foundation + one resonator (flattened for display)."""
    top = gdstk.Cell(f"preview_{inst_name}")
    add_foundation_refs(top, frame_cell, ppd_cell, foundation)
    rdx, rdy = placement_shift(res, frame_cell, ppd_cell, foundation=foundation)
    top.add(
        gdstk.Reference(
            res.reference.cell,
            origin=(res.origin[0] + rdx, res.origin[1] + rdy),
            rotation=res.rotation,
            magnification=res.magnification,
            x_reflection=res.x_reflection,
        )
    )
    return top.flatten()


print("\nFoundation previews (frame + centered ppd + resonator):")
for i, res in enumerate(res_list):
    inst = inst_names[i]
    flat = preview_rteg_cell(res, inst)
    with tempfile.TemporaryDirectory() as tmp:
        svg_path = Path(tmp) / f"{inst}.svg"
        flat.write_svg(str(svg_path))
        print(f"\n[{i}] {inst} ({res.res_type}) — {rteg_cell_name(parent, inst)}")
        display(SVG(svg_path.read_text(encoding="utf-8")))


## 6. `build_rteg.py` — visual export (no preserved metal)

One GDS per resonator using the same foundation: frame at top-left, ppd centered in frame, resonator centered on the assembly. Quick KLayout check that separation picked the right geometry.

In [ ]:
from build_rteg import export_resonators

build_dir = DRAFT / "notebook_build"
build_stats = export_resonators(FILTER, build_dir)

pd.DataFrame([
    {
        "cell": s.cell_name,
        "inst": s.inst_name,
        "type": s.res_type,
        "filter@": s.filter_origin,
        "rteg@": tuple(round(x, 1) for x in s.rteg_origin),
    }
    for s in build_stats
])

## 7. `prepare_rteg.py` — routing input assembly

Builds the full pre-route cell:
1. Die frame at top-left; `ppd_1port` centered in frame (`build_foundation`)
2. Resonator bbox center aligned to assembly center
3. Preserved connect metal + nearby vias (same shift as resonator)
4. Trim to golden layer set (drops BF/TSV/EM_VPT fill)

Below: prepare **golden anchor S3** (filter index 6).

In [ ]:
from prepare_rteg import prepare_resonator, prepare_all

S3_INDEX = 6

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    prepared_path, prep_stats = prepare_resonator(S3_INDEX, golden_gds=GOLDEN, output_dir=DRAFT)

print(f"Output: {prepared_path.name}")
print(f"  inst_name={prep_stats.inst_name}")
print(f"  filter@={tuple(round(x,1) for x in prep_stats.filter_origin)}")
print(f"  rteg@={tuple(round(x,1) for x in prep_stats.rteg_origin)}  (resonator -> assembly center)")
print(f"  metal: {prep_stats.preserved_poly_count} polys ({prep_stats.metal_source})")
print(f"  vias={prep_stats.via_count}  trimmed={prep_stats.trimmed_poly_count} polys")
print(f"  layers absent from golden: {prep_stats.layers_absent_from_golden or 'none'}")
for msg in w:
    print(f"  WARNING: {msg.message}")

# Verify top-cell structure
prep_lib = gdstk.read_gds(prepared_path)
top = next(c for c in prep_lib.cells if c.name == prep_stats.cell_name)
print(f"\nTop cell bbox: {top.bounding_box()}")
print(f"References ({len(top.references)}):")
for ref in top.references:
    name = ref.cell.name if ref.cell else "?"
    print(f"  {name[:36]:36s} @ {tuple(round(x,1) for x in ref.origin)}")

In [ ]:
# Optional: batch all 8 resonators (same as `python prepare_rteg.py --all`)
RUN_BATCH = False  # set True to regenerate all prepared files

if RUN_BATCH:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        batch = prepare_all(golden_gds=GOLDEN, output_dir=DRAFT)
    pd.DataFrame([
        {"file": p.name, "inst": s.inst_name, "rteg@": tuple(round(x,1) for x in s.rteg_origin)}
        for p, s in batch
    ])
else:
    print("Set RUN_BATCH=True to run prepare_all() for all 8 resonators.")

## 8. `inspect_golden.py` — prepared vs golden diff

Read-only diagnostic: layer inventory, MBE/MTE extents, and what the golden has that prepare does not yet generate (fill/trim layers).

In [ ]:
from inspect_golden import build_notes, grouped_missing_layers

notes = build_notes(GOLDEN, prepared_path)
print(notes)

missing = grouped_missing_layers(GOLDEN, prepared_path)
pd.DataFrame(missing) if missing else "No grouped missing layers."

## 9. `geometry.py` — shared boolean / net helpers

Not run directly. Used by `prepare_rteg.py` (layer trim) and `route_rteg.py` (routable region, connectors, DRC nets).

In [ ]:
import geometry as G

prep_top = gdstk.read_gds(prepared_path).cells[-1]
flat = G.flatten_cell(prep_top)
print(f"Flattened prepared top: {len(flat)} polygons")

# Layer trim helper (same allow-list as prepare)
from route_rteg import resolve_allowed_layer_pairs, RouteConfig
allowed = resolve_allowed_layer_pairs(RouteConfig(), GOLDEN)
print(f"Golden allow-list: {len(allowed)} layer pairs")

## 10. `route_rteg.py` — routing v1 (S3 demo)

Draws a simple signal connector, recuts frame ground, runs net-aware DRC, writes routed GDS + net overlay + report.

Set `RUN_ROUTE = True` to execute (takes a few seconds). Uses the SKILL-named prepared file from step 7.

In [ ]:
RUN_ROUTE = True  # set False to skip routing

if RUN_ROUTE:
    from route_rteg import route, write_report, RouteConfig

    prepared_path = Path(prepared_path) if "prepared_path" in globals() else DRAFT / "KB331_N_01_RTEG1_S3_prepared.gds"
    if not prepared_path.is_file():
        raise FileNotFoundError(f"Missing {prepared_path} — run the prepare cell (section 7) first.")

    cfg = RouteConfig()
    result = route(prepared_path, GOLDEN, cfg)
    report_path = write_report(result, cfg, prepared_path, GOLDEN)

    print("=== Route result ===")
    print(f"Prepared:    {prepared_path.name}")
    print(f"Routed GDS:  {result.routed_path}")
    print(f"Net overlay: {result.nets_path}")
    print(f"Report:      {report_path}")
    print(f"Signal route drawn: {result.signal_routed} ({result.signal_kind or 'n/a'})")
    print(f"DRC introduced violations: {result.real_drc_violations}")
    for layer in ("BAW_MBE", "BAW_MTE"):
        pct = result.overlap.get(layer, {}).get("overlap_fraction_of_golden", 0) * 100
        print(f"{layer} overlap vs golden: {pct:.1f}%")
else:
    print("Routing skipped (RUN_ROUTE=False).")

## 11. Output files in `draft_output/`

Open GDS in **KLayout**. Use `*_nets.gds` with `*_nets.lyp` to visualize net classification from routing.

In [ ]:
DRAFT.mkdir(parents=True, exist_ok=True)

outputs = sorted(DRAFT.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
rows = []
for p in outputs[:25]:
    kind = "GDS" if p.suffix == ".gds" else p.suffix.lstrip(".") or "file"
    rows.append({"name": p.name, "kind": kind, "size_KB": round(p.stat().st_size / 1024, 1)})

pd.DataFrame(rows) if rows else "No files in draft_output yet."

## Quick reference — CLI equivalents

```powershell
cd python_code
python layermap.py
python inspect_refs.py
python separate.py
python build_rteg.py
python prepare_rteg.py --all
python inspect_golden.py --prepared draft_output/KB331_N_01_RTEG1_S3_prepared.gds
python route_rteg.py --prepared draft_output/KB331_N_01_RTEG1_S3_prepared.gds
```

**Open in KLayout:**
- `draft_output/KB331_N_01_RTEG1_S3_prepared.gds` — pre-route SKILL assembly
- `draft_output/KB331_N_01_RTEG1_*_routed.gds` — routed draft (if step 10 ran)
- `example_output/KB331_N_01_RTEG1_S3.gds` — golden reference